# 🧬 Ensemble 4 model: YOLO11 + YOLO26 (CNN) + RT-DETR + RF-DETR (Transformer)

Gộp các model khác kiến trúc bằng **Weighted Box Fusion** → bù trừ lỗi → kỳ vọng mAP cao hơn từng model.

**Cần các file weights (đã lưu Drive):**
`yolo11s_best.pt`, `yolo26s_best.pt`, `rtdetr_best.pt`, `rfdetr_small_best.pth` + `summary_rfdetr.json`

**File này sẽ:**
1. Nạp đủ 4 model (thiếu cái nào tự bỏ qua)
2. Gộp WBF (weight = mAP50-95 đã đo: model mạnh đóng góp nhiều hơn)
3. **Tính mAP THẬT** của ensemble vs từng model trên tập valid → cho biết có vượt 73.5% (YOLO11s) không

## Xử lý sẵn 3 khác biệt khi gộp đa kiến trúc
1. **Khác framework**: YOLO dùng `ultralytics`, RF-DETR dùng `rfdetr` → 2 hàm predict riêng.
2. **Khác thứ tự class**: gộp theo **TÊN class** (không theo số id) — RF-DETR coco hay lệch id.
3. **Khác toạ độ**: chuẩn hoá hết về xyxy [0,1] rồi mới WBF.

`Runtime → T4 GPU → Run all`

In [ ]:
# 1 — Cài đặt + mount Drive
!nvidia-smi -L
%pip install -q -U "ultralytics>=8.4.0" rfdetr supervision roboflow ensemble-boxes
import os, json, glob
from pathlib import Path
import numpy as np, cv2
import matplotlib.pyplot as plt
from ensemble_boxes import weighted_boxes_fusion

from google.colab import drive
drive.mount('/content/drive')
OUT  = Path('/content/drive/MyDrive/DrowsyDriver_Results')
HOME = Path('/content')

# Class space CHUẨN (thứ tự yolo data.yaml)
UNIFIED = ['close_eyeL','close_eyeR','no_yawn','open_eyeL','open_eyeR','yawn']
UNI_IDX = {n:i for i,n in enumerate(UNIFIED)}
I_CL,I_CR = UNI_IDX['close_eyeL'], UNI_IDX['close_eyeR']
I_OL,I_OR = UNI_IDX['open_eyeL'],  UNI_IDX['open_eyeR']
I_YAWN    = UNI_IDX['yawn']

In [ ]:
# 2 — Load 4 model (thiếu cái nào tự bỏ qua). Weight = mAP50-95 đã đo (model mạnh "nói to" hơn)
from ultralytics import YOLO, RTDETR

MODELS = []   # (tên, obj, kind, weight)
if (OUT/'yolo11s_best.pt').exists():
    MODELS.append(('YOLO11',  YOLO(str(OUT/'yolo11s_best.pt')),  'ultra', 0.735)); print('✅ YOLO11')
if (OUT/'yolo26s_best.pt').exists():
    MODELS.append(('YOLO26',  YOLO(str(OUT/'yolo26s_best.pt')),  'ultra', 0.714)); print('✅ YOLO26')
if (OUT/'rtdetr_best.pt').exists():
    MODELS.append(('RT-DETR', RTDETR(str(OUT/'rtdetr_best.pt')), 'ultra', 0.707)); print('✅ RT-DETR')
if (OUT/'rfdetr_small_best.pth').exists():     # ← .pth (không phải .pt)
    from rfdetr import RFDETRSmall
    MODELS.append(('RF-DETR', RFDETRSmall(pretrain_weights=str(OUT/'rfdetr_small_best.pth')), 'rfdetr', 0.673)); print('✅ RF-DETR')

print(f'\n  {len(MODELS)} model sẽ ensemble:', [m[0] for m in MODELS])
assert len(MODELS) >= 2, '❌ Cần ít nhất 2 model — kiểm tra file weights trên Drive'

In [ ]:
# 3 — Bản đồ class cho RF-DETR (coco) + ảnh test
# RF-DETR predict ra class_id theo thứ tự category COCO → map sang TÊN
RFDETR_NAMES = []
sp = OUT/'summary_rfdetr.json'
if sp.exists():
    RFDETR_NAMES = json.loads(sp.read_text()).get('classes', [])
print('  RFDETR_NAMES (theo id):', RFDETR_NAMES)
print('  → Nếu list này có 1 tên lạ ở đầu (vd "drowsy"/"objects") thì đó là supercategory, OK,')
print('    nó sẽ tự bị bỏ qua vì không khớp tên trong UNIFIED.')

# Tải ảnh test
from roboflow import Roboflow
rf = Roboflow(api_key='qI3lEKlNpIZpNENdk3MH')
ds = None
for proj in ['datio_yolo','driver-yawn','driver-yawn-wh6wj']:
    try: ds = rf.workspace('nguyen-tuan-dat').project(proj).version(1).download('yolov11'); break
    except Exception: pass
base = Path(ds.location)
TEST_DIR = base/'test'/'images' if (base/'test'/'images').exists() else base/'valid'/'images'
test_imgs = sorted(list(TEST_DIR.glob('*.jpg'))+list(TEST_DIR.glob('*.png')))
print(f'  Test images: {len(test_imgs)}')

In [ ]:
# 4 — Hàm predict THỐNG NHẤT: trả (boxes xyxy[0,1], scores, unified_class_id)
def predict_ultra(model, img_path, conf=0.01):
    r = model.predict(img_path, conf=conf, iou=0.7, verbose=False)[0]
    if len(r.boxes)==0: return [],[],[]
    bn = r.boxes.xyxyn.cpu().numpy().tolist()
    sc = r.boxes.conf.cpu().numpy().tolist()
    nm = [model.names[int(c)] for c in r.boxes.cls.cpu().numpy()]
    out = [(b,s,UNI_IDX[n]) for b,s,n in zip(bn,sc,nm) if n in UNI_IDX]
    return list(map(list, zip(*out))) if out else ([],[],[])

def predict_rfdetr(model, img_path, conf=0.01):
    det = model.predict(img_path, threshold=conf)
    if len(det)==0: return [],[],[]
    h,w = cv2.imread(img_path).shape[:2]
    out = []
    for (x1,y1,x2,y2), s, c in zip(det.xyxy, det.confidence, det.class_id):
        nm = RFDETR_NAMES[int(c)] if int(c) < len(RFDETR_NAMES) else None
        if nm in UNI_IDX:
            out.append(([x1/w,y1/h,x2/w,y2/h], float(s), UNI_IDX[nm]))
    return list(map(list, zip(*out))) if out else ([],[],[])

def predict_one(name, model, kind, img_path):
    return (predict_rfdetr if kind=='rfdetr' else predict_ultra)(model, img_path)

# KIỂM TRA căn chỉnh class — in tên class mỗi model dự đoán trên 1 ảnh
print('  Sanity-check tên class trên ảnh test[0]:')
for name, model, kind, _ in MODELS:
    b,s,l = predict_one(name, model, kind, str(test_imgs[0]))
    seen = sorted({UNIFIED[i] for i in l})
    print(f'    {name:<8}: {seen if seen else "(không có box >0.01 — thử ảnh khác)"}')
print('  → 3 model nên cho ra tên class trong cùng tập 6 tên. Nếu RF-DETR ra tên LẠ → sửa RFDETR_NAMES.')

In [ ]:
# 5 — ENSEMBLE bằng Weighted Box Fusion
def ensemble(img_path, iou=0.55, skip=0.25):
    B,S,L,W = [],[],[],[]
    for name, model, kind, w in MODELS:
        b,s,l = predict_one(name, model, kind, img_path)
        if b: B.append(b); S.append(s); L.append(l); W.append(w)
    if not B: return [],[],[]
    return weighted_boxes_fusion(B, S, L, weights=W, iou_thr=iou, skip_box_thr=skip)

# Demo 6 ảnh
COLORS = {I_CL:(255,80,80),I_CR:(255,160,0),I_OL:(80,160,255),I_OR:(0,200,120),I_YAWN:(220,0,220)}
fig, ax = plt.subplots(2,3, figsize=(16,9))
for a, ip in zip(ax.flat, test_imgs[:6]):
    img = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB); h,w = img.shape[:2]
    bxs,scs,lbs = ensemble(str(ip))
    for bx,sc,lb in zip(bxs,scs,lbs):
        if sc<0.3: continue
        x1,y1,x2,y2 = int(bx[0]*w),int(bx[1]*h),int(bx[2]*w),int(bx[3]*h)
        c = COLORS.get(int(lb),(255,255,0))
        cv2.rectangle(img,(x1,y1),(x2,y2),c,2)
        cv2.putText(img,f'{UNIFIED[int(lb)]} {sc:.2f}',(x1,max(12,y1-5)),cv2.FONT_HERSHEY_SIMPLEX,0.45,c,1)
    a.imshow(img); a.axis('off')
plt.suptitle(f'Ensemble WBF ({" + ".join(m[0] for m in MODELS)})', fontweight='bold')
plt.tight_layout(); plt.savefig(OUT/'ensemble_3model.png', dpi=120, bbox_inches='tight'); plt.show()
print('✅  Lưu', OUT/'ensemble_3model.png')

In [ ]:
# 6 — Drowsiness Decision Layer: 6 class → 1 điểm buồn ngủ
def max_conf_per_class(bxs,scs,lbs,thr=0.25):
    out={i:0.0 for i in range(len(UNIFIED))}
    for s,l in zip(scs,lbs):
        if s>=thr: out[int(l)]=max(out[int(l)],float(s))
    return out

def drowsiness_score(conf):
    cL,cR=conf[I_CL],conf[I_CR]; oL,oR=conf[I_OL],conf[I_OR]; y=conf[I_YAWN]
    eye=( cL/(cL+oL+1e-6) + cR/(cR+oR+1e-6) )/2
    s=0.6*eye+0.4*y
    st='DROWSY 🔴' if s>=0.6 else ('WARNING 🟡' if s>=0.35 else 'ALERT 🟢')
    return s, st

print(f'  {"image":<24}{"score":>7}  state')
for ip in test_imgs[:12]:
    b,s,l = ensemble(str(ip))
    sc, st = drowsiness_score(max_conf_per_class(b,s,l))
    print(f'  {ip.name[:24]:<24}{sc:7.2f}  {st}')

In [ ]:
# 7 — mAP THẬT: từng model vs ENSEMBLE trên tập valid (supervision, cùng thước đo)
from supervision.metrics import MeanAveragePrecision

split = 'valid' if (base/'valid'/'images').exists() else 'test'
gt = sv.DetectionDataset.from_yolo(
    images_directory_path=str(base/split/'images'),
    annotations_directory_path=str(base/split/'labels'),
    data_yaml_path=str(base/'data.yaml'))
print(f'  Chấm trên "{split}": {len(gt)} ảnh — chạy mỗi model 1 lượt...')

def to_det(bxs, scs, lbs, w, h):
    if not bxs: return sv.Detections.empty()
    xyxy = np.array(bxs, dtype=float) * np.array([w, h, w, h])
    return sv.Detections(xyxy=xyxy, confidence=np.array(scs, dtype=float),
                         class_id=np.array(lbs).astype(int))

# Chạy MỖI model đúng 1 lượt rồi cache (ensemble ghép lại từ cache, không chạy lại)
cache = {name: [] for name,_,_,_ in MODELS}
meta  = []
for path, image, ann in gt:
    h, w = image.shape[:2]; meta.append((w, h, ann))
    for name, model, kind, _ in MODELS:
        cache[name].append(predict_one(name, model, kind, path))

def ens_at(i):
    B,S,L,W = [],[],[],[]
    for name,_,_,wt in MODELS:
        b,s,l = cache[name][i]
        if b: B.append(b); S.append(s); L.append(l); W.append(wt)
    if not B: return [],[],[]
    return weighted_boxes_fusion(B,S,L, weights=W, iou_thr=0.55, skip_box_thr=0.25)

targs = [m[2] for m in meta]
def mAP(preds):
    r = MeanAveragePrecision().update(preds, targs).compute()
    return r.map50*100, r.map50_95*100

print('\n  Model                | mAP50  | mAP50-95')
print('  ' + '-'*44)
best_single = 0.0
for name,_,_,_ in MODELS:
    preds = [to_det(*cache[name][i], meta[i][0], meta[i][1]) for i in range(len(meta))]
    m50, m5095 = mAP(preds); best_single = max(best_single, m5095)
    print(f'  {name:<20} | {m50:5.2f}% | {m5095:5.2f}%')

ens_preds = [to_det(*ens_at(i), meta[i][0], meta[i][1]) for i in range(len(meta))]
em50, em5095 = mAP(ens_preds)
print(f'  {"ENSEMBLE (WBF)":<20} | {em50:5.2f}% | {em5095:5.2f}%')
print(f'\n  Single tốt nhất mAP50-95 = {best_single:.2f}%  →  ENSEMBLE = {em5095:.2f}%  '
      + ('✅ VƯỢT!' if em5095 > best_single else '➖ chưa vượt — thử chỉnh weights / iou_thr / skip_box_thr'))

---
## 🔬 Transformer khác có thể thử thêm

| Model | Trạng thái | Cách thêm vào ensemble |
|-------|-----------|------------------------|
| **RT-DETR** | ✅ đã có | `from ultralytics import RTDETR` (file này dùng rồi) |
| **RT-DETRv2** | ultralytics hỗ trợ | đổi weights `rtdetr-v2-*.pt` trong file RT-DETR |
| **RF-DETR Nano/Medium** | rfdetr | đổi `RFDETRSmall` → `RFDETRMedium` (mAP cao hơn, chậm hơn) |
| **D-FINE** | repo riêng (SOTA 2024) | clone github D-FINE, export ONNX rồi wrap `predict_*` |
| **DEIM** | repo riêng | tương tự D-FINE, cải tiến matching |

**Khuyến nghị cho bài này:** YOLO26 + RT-DETR + RF-DETR là đủ mạnh & dễ chạy trên T4.  
D-FINE/DEIM mAP nhỉnh hơn chút nhưng phải setup repo riêng (không có trong ultralytics/rfdetr) → chỉ nên thử nếu còn thời gian.

**Để thêm 1 transformer mới vào ensemble:** chỉ cần viết 1 hàm `predict_xxx(model, img_path)` trả về
`(boxes xyxy[0,1], scores, unified_class_id)` rồi `MODELS.append((tên, model, 'xxx', weight))` — phần WBF tự lo.

### Vì sao CNN + Transformer bù trừ nhau?
- **YOLO26 (CNN)**: nhanh, giỏi vật thể nhỏ/rõ, nhưng dễ trượt khi bị che khuất.
- **RF-DETR/RT-DETR (Transformer)**: attention toàn cục → tốt khi mặt nghiêng/che một phần, định vị box chính xác hơn.
- Gộp lại: box nào **cả 2 loại cùng đồng ý** → độ tin cậy rất cao → giảm cả false positive lẫn false negative.